# B3b Defense – 02 Macro Data Processing & Stationarity

**Objective:** Load all three raw CSVs, resample FDEFX from quarterly to monthly,
build a shared monthly index, run ADF stationarity tests, apply first differencing,
and save `macro_features_defense.csv` to `data/processed/`.

No API calls – all data is loaded from `data/raw/`.

**Variable roles after target redesign:**
- **FDEFX** – *Target variable*: Federal Defense Consumption Expenditures (realized spending, USD millions). Chosen for semantic consistency with B1/B2 invoice-based forecasts in SAC.
- **ADEFNO** – *Leading feature* (lags 1–24): Manufacturers' New Orders: Defense Capital Goods. Captures order-to-bill conversion dynamics; the 24-month lag window reflects heterogeneous lead times across Defense Capital Goods categories.
- **IPB52300S** – *Coincident feature* (lags 1–12): Industrial Production: Defense and Space Equipment. Reflects production capacity.

**Output columns:** `date, ADEFNO, ADEFNO_diff, IPB52300S, IPB52300S_diff, FDEFX, FDEFX_diff`

In [1]:
import os
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller

DATA_RAW       = '../data/raw/'
DATA_PROCESSED = '../data/processed/'

os.makedirs(DATA_PROCESSED, exist_ok=True)

## Section 1: Load Data

Same filter as Notebook 01: **2000-01 to 2025-12**  
(Structural break after 9/11 – see Notebook 01 for rationale.)

In [2]:
# Load ADEFNO (monthly)
adefno = pd.read_csv(DATA_RAW + 'ADEFNO.csv', parse_dates=['observation_date'])
adefno = adefno.rename(columns={'observation_date': 'date'})
adefno = adefno[(adefno['date'] >= '2000-01-01') & (adefno['date'] <= '2025-12-31')].copy()
adefno = adefno.sort_values('date').set_index('date')

# Load IPB52300S (monthly)
ipb = pd.read_csv(DATA_RAW + 'IPB52300S.csv', parse_dates=['observation_date'])
ipb = ipb.rename(columns={'observation_date': 'date'})
ipb = ipb[(ipb['date'] >= '2000-01-01') & (ipb['date'] <= '2025-12-31')].copy()
ipb = ipb.sort_values('date').set_index('date')

# Load FDEFX (quarterly)
fdefx_raw = pd.read_csv(DATA_RAW + 'FDEFX.csv', parse_dates=['observation_date'])
fdefx_raw = fdefx_raw.rename(columns={'observation_date': 'date'})
fdefx_raw = fdefx_raw[(fdefx_raw['date'] >= '2000-01-01') & (fdefx_raw['date'] <= '2025-12-31')].copy()
fdefx_raw = fdefx_raw.sort_values('date').set_index('date')

print('ADEFNO shape (monthly):  ', adefno.shape)
print('IPB52300S shape (monthly):', ipb.shape)
print('FDEFX shape (quarterly): ', fdefx_raw.shape)

ADEFNO shape (monthly):   (312, 1)
IPB52300S shape (monthly): (312, 1)
FDEFX shape (quarterly):  (104, 1)


## Section 2: FDEFX Quarterly → Monthly (Forward-Fill)

**Method: forward-fill** (carry the last known quarterly value forward to fill intermediate months).

Forward-fill is preferred over linear interpolation to avoid look-ahead bias:
interpolation would use the *next* quarterly observation to fill intermediate months,
which is information that was not yet available in those months.
Forward-fill only uses the *last known* value – no future information leaks into the past.

> **Design decision & model limitation:** FDEFX is disaggregated from quarterly to monthly
> frequency via forward-fill. This is a deliberate choice to align the data grid with the
> monthly ADEFNO and IPB52300S series. However, the effective information density of FDEFX
> remains quarterly — three consecutive months share the identical value. Downstream lag and
> rolling-window features built on FDEFX inherit this stepped pattern. This is documented as
> a model limitation and should be noted in any external presentation of forecast results.

In [3]:
monthly_index = pd.date_range('2000-01-01', '2025-12-01', freq='MS')

# Reindex to monthly, then forward-fill
fdefx_monthly = fdefx_raw.reindex(monthly_index).ffill()
fdefx_monthly.index.name = 'date'

print('FDEFX monthly shape after forward-fill:', fdefx_monthly.shape)
print(f'Expected ~{len(monthly_index)} rows')
print()
print('First 6 rows (shows quarterly value repeated for 3 months):')
print(fdefx_monthly.head(6))

FDEFX monthly shape after forward-fill: (312, 1)
Expected ~312 rows

First 6 rows (shows quarterly value repeated for 3 months):
              FDEFX
date               
2000-01-01  383.028
2000-02-01  383.028
2000-03-01  383.028
2000-04-01  399.592
2000-05-01  399.592
2000-06-01  399.592


## Section 3: Build Common Monthly Index

In [4]:
# Inner join on shared monthly DatetimeIndex
df_combined = adefno.join(ipb, how='inner').join(fdefx_monthly, how='inner')
df_combined.columns = ['ADEFNO', 'IPB52300S', 'FDEFX']

print('df_combined shape:', df_combined.shape)
print(f'Date range: {df_combined.index.min().date()} to {df_combined.index.max().date()}')
print()
print('Missing values after join:')
print(df_combined.isnull().sum())
print()
df_combined.head()

df_combined shape: (312, 3)
Date range: 2000-01-01 to 2025-12-01

Missing values after join:
ADEFNO       0
IPB52300S    0
FDEFX        0
dtype: int64



,ADEFNO,IPB52300S,FDEFX
date,,,
2000-01-01,7159,71.0321,383.028
2000-02-01,5162,69.3458,383.028
2000-03-01,3773,68.6844,383.028
2000-04-01,3568,67.3689,399.592
2000-05-01,5023,66.5616,399.592


## Section 4: ADF Stationarity Tests

The Augmented Dickey-Fuller (ADF) test checks whether a series has a unit root
(i.e., is non-stationary). H₀: series has a unit root (non-stationary).
Reject H₀ if p < 0.05 → stationary.

**Expected outcome:**
- ADEFNO and FDEFX likely non-stationary (upward trend visible in EDA)
- IPB52300S may also be non-stationary (gradual trend)

**Note on FDEFX as target:** The ADF test on `FDEFX_diff` (see output below) yields p=0.2193,
meaning FDEFX does not achieve stationarity even after first differencing. This is a known
consequence of the forward-fill disaggregation — repeated identical values suppress the
variance that ADF relies on. **FDEFX as target will be used in absolute level (not differenced)**,
since XGBoost does not require stationarity of the target variable. The differenced series
(`FDEFX_diff`) is still computed and provided as a feature in Notebook 03 to capture
short-term expenditure dynamics.

In [5]:
def run_adf(series, name):
    """Run ADF test and return result dict."""
    clean = series.dropna()
    stat, pval, _, _, _, _ = adfuller(clean, autolag='AIC')
    return {
        'Series':        name,
        'ADF Statistic': round(stat, 4),
        'p-value':       round(pval, 4),
        'Stationary (p<0.05)': pval < 0.05,
    }

In [6]:
adf_levels = pd.DataFrame([
    run_adf(df_combined['ADEFNO'],    'ADEFNO (level)'),
    run_adf(df_combined['IPB52300S'], 'IPB52300S (level)'),
    run_adf(df_combined['FDEFX'],     'FDEFX (level, ffill)'),
])

print('=== ADF Tests – Levels ===')
print(adf_levels.to_string(index=False))

=== ADF Tests – Levels ===
              Series  ADF Statistic  p-value  Stationary (p<0.05)
      ADEFNO (level)        -0.2886   0.9271                False
   IPB52300S (level)        -2.4630   0.1247                False
FDEFX (level, ffill)        -0.8613   0.8004                False


## Section 5: First Differencing

In [7]:
# Compute first differences for all three series
df_combined['ADEFNO_diff']    = df_combined['ADEFNO'].diff()
df_combined['IPB52300S_diff'] = df_combined['IPB52300S'].diff()
df_combined['FDEFX_diff']     = df_combined['FDEFX'].diff()

# Re-run ADF on first differences
adf_diffs = pd.DataFrame([
    run_adf(df_combined['ADEFNO_diff'],    'ADEFNO_diff'),
    run_adf(df_combined['IPB52300S_diff'], 'IPB52300S_diff'),
    run_adf(df_combined['FDEFX_diff'],     'FDEFX_diff'),
])

print('=== ADF Tests – First Differences ===')
print(adf_diffs.to_string(index=False))

=== ADF Tests – First Differences ===
        Series  ADF Statistic  p-value  Stationary (p<0.05)
   ADEFNO_diff        -8.3388   0.0000                 True
IPB52300S_diff        -7.7546   0.0000                 True
    FDEFX_diff        -2.1646   0.2193                False


**ADF result interpretation for FDEFX_diff (p=0.2193):**
FDEFX_diff is not stationary after first differencing. This is expected: forward-fill produces
three identical monthly values per quarter, which suppresses variance and inflates the ADF
p-value. As noted in the Section 4 header, FDEFX will be used as target in absolute level.
XGBoost does not require the target to be stationary — tree-based splits are scale-invariant
and level-sensitive. `FDEFX_diff` is retained as a feature to capture quarter-over-quarter
expenditure momentum.

## Section 6: Save Output

In [8]:
# Build output DataFrame with explicit column order
df_output = df_combined[[
    'ADEFNO', 'ADEFNO_diff',
    'IPB52300S', 'IPB52300S_diff',
    'FDEFX', 'FDEFX_diff',
]].copy()

# Drop the single NaN row introduced by .diff() on the first observation
df_output = df_output.dropna()

# Reset index and format date as YYYY-MM string
df_output = df_output.reset_index()
df_output['date'] = df_output['date'].dt.strftime('%Y-%m')

out_path = DATA_PROCESSED + 'macro_features_defense.csv'
df_output.to_csv(out_path, index=False)

print(f'Saved: {out_path}')
print(f'Shape: {df_output.shape}')
print()
print(df_output.head())

Saved: ../data/processed/macro_features_defense.csv
Shape: (311, 7)

      date  ADEFNO  ADEFNO_diff  IPB52300S  IPB52300S_diff    FDEFX  \
0  2000-02    5162      -1997.0    69.3458         -1.6863  383.028   
1  2000-03    3773      -1389.0    68.6844         -0.6614  383.028   
2  2000-04    3568       -205.0    67.3689         -1.3155  399.592   
3  2000-05    5023       1455.0    66.5616         -0.8073  399.592   
4  2000-06   25893      20870.0    67.1028          0.5412  399.592   

   FDEFX_diff  
0       0.000  
1       0.000  
2      16.564  
3       0.000  
4       0.000  
